# Graph Query Languages: SPARQL vs Cypher

Two dominant paradigms for graph databases:

| | **RDF / SPARQL** (e.g., Apache Jena) | **LPG / Cypher** (e.g., Neo4j) |
|---|---|---|
| Data model | Triples `(s, p, o)` with IRIs | Nodes & relationships with properties |
| Semantics | Strong (RDFS/OWL reasoning) | Implicit (labels, conventions) |
| Edge properties | Via reification (extra triples) | Native (first-class) |
| Best for | Semantic interoperability, reasoning | Fast traversals, graph analytics |

This notebook demonstrates both with the **same agentic AI scenario**, using rdflib for live SPARQL queries and showing equivalent Cypher syntax.

In [ ]:
%pip install rdflib -q

In [ ]:
from rdflib import Graph, Namespace, Literal, URIRef
from rdflib.namespace import RDF, RDFS, XSD

## 1. The Agentic AI Scenario

We model the same data in both paradigms:
- **Agents**: AgentSummarizer (LLMAgent), AgentRetriever (RetrievalAgent)
- **Tasks**: Task123 (ResearchTask, pending), Task456 (SearchTask, completed)
- **Documents**: Doc1, Doc2, Doc3
- **Relationships**: assignedTo, hasInput, hasCapability, hasStatus

### RDF Data (Turtle)

```turtle
@prefix ex: <http://example.org/> .

ex:Task123 a ex:ResearchTask ;
    ex:assignedTo ex:AgentSummarizer ;
    ex:hasInput ex:Doc1 ;
    ex:hasInput ex:Doc2 ;
    ex:hasStatus "pending" .

ex:Task456 a ex:SearchTask ;
    ex:assignedTo ex:AgentRetriever ;
    ex:hasInput ex:Doc3 ;
    ex:hasStatus "completed" .

ex:AgentSummarizer a ex:LLMAgent ;
    ex:hasCapability "summarization" ;
    ex:hasCapability "writing" .

ex:AgentRetriever a ex:RetrievalAgent ;
    ex:hasCapability "search" ;
    ex:hasCapability "indexing" .
```

In [ ]:
# Build the RDF graph
g = Graph()
EX = Namespace("http://example.org/")
g.bind("ex", EX)

# Tasks
g.add((EX.Task123, RDF.type, EX.ResearchTask))
g.add((EX.Task123, EX.assignedTo, EX.AgentSummarizer))
g.add((EX.Task123, EX.hasInput, EX.Doc1))
g.add((EX.Task123, EX.hasInput, EX.Doc2))
g.add((EX.Task123, EX.hasStatus, Literal("pending")))

g.add((EX.Task456, RDF.type, EX.SearchTask))
g.add((EX.Task456, EX.assignedTo, EX.AgentRetriever))
g.add((EX.Task456, EX.hasInput, EX.Doc3))
g.add((EX.Task456, EX.hasStatus, Literal("completed")))

# Agents
g.add((EX.AgentSummarizer, RDF.type, EX.LLMAgent))
g.add((EX.AgentSummarizer, EX.hasCapability, Literal("summarization")))
g.add((EX.AgentSummarizer, EX.hasCapability, Literal("writing")))

g.add((EX.AgentRetriever, RDF.type, EX.RetrievalAgent))
g.add((EX.AgentRetriever, EX.hasCapability, Literal("search")))
g.add((EX.AgentRetriever, EX.hasCapability, Literal("indexing")))

# Documents
g.add((EX.Doc1, RDF.type, EX.Document))
g.add((EX.Doc1, EX.title, Literal("Research Paper on Transformers")))
g.add((EX.Doc2, RDF.type, EX.Document))
g.add((EX.Doc2, EX.title, Literal("Dataset: NLP Benchmarks")))
g.add((EX.Doc3, RDF.type, EX.Document))
g.add((EX.Doc3, EX.title, Literal("Web Search Results")))

print(f"RDF graph: {len(g)} triples")
print("\nTurtle serialization:")
print(g.serialize(format="turtle"))

### Equivalent Cypher (Neo4j)

The same data in Cypher's labeled property graph model:

```cypher
// Create agents
CREATE (a1:Agent:LLMAgent {id: "AgentSummarizer", capabilities: ["summarization", "writing"]})
CREATE (a2:Agent:RetrievalAgent {id: "AgentRetriever", capabilities: ["search", "indexing"]})

// Create tasks
CREATE (t1:Task:ResearchTask {id: "Task123", status: "pending"})
CREATE (t2:Task:SearchTask {id: "Task456", status: "completed"})

// Create documents
CREATE (d1:Document {id: "Doc1", title: "Research Paper on Transformers"})
CREATE (d2:Document {id: "Doc2", title: "Dataset: NLP Benchmarks"})
CREATE (d3:Document {id: "Doc3", title: "Web Search Results"})

// Create relationships
CREATE (t1)-[:ASSIGNED_TO]->(a1)
CREATE (t1)-[:HAS_INPUT]->(d1)
CREATE (t1)-[:HAS_INPUT]->(d2)
CREATE (t2)-[:ASSIGNED_TO]->(a2)
CREATE (t2)-[:HAS_INPUT]->(d3)
```

Notice how Cypher stores capabilities as a **property array** on the node, while RDF uses separate triples.

## 2. Side-by-Side Query Comparison

For each query pattern, we show **SPARQL** (executed live) and the equivalent **Cypher**.

### Query A: Find all pending tasks and their assigned agents

In [ ]:
# SPARQL
sparql_a = """
PREFIX ex: <http://example.org/>

SELECT ?task ?agent
WHERE {
    ?task ex:hasStatus "pending" .
    ?task ex:assignedTo ?agent .
}
"""

print("SPARQL: Find pending tasks and their agents")
print("=" * 50)
for row in g.query(sparql_a):
    task = str(row.task).split("/")[-1]
    agent = str(row.agent).split("/")[-1]
    print(f"  Task: {task}  →  Agent: {agent}")

print("\n--- Equivalent Cypher ---")
print("""
MATCH (t:Task {status: "pending"})-[:ASSIGNED_TO]->(a:Agent)
RETURN t.id AS task, a.id AS agent
""")

### Query B: Find agents with a specific capability

In [ ]:
# SPARQL
sparql_b = """
PREFIX ex: <http://example.org/>

SELECT ?agent ?type ?capability
WHERE {
    ?agent ex:hasCapability ?capability .
    ?agent a ?type .
    FILTER(?capability = "summarization")
}
"""

print("SPARQL: Find agents with 'summarization' capability")
print("=" * 50)
for row in g.query(sparql_b):
    agent = str(row.agent).split("/")[-1]
    atype = str(row.type).split("/")[-1]
    print(f"  Agent: {agent} (type: {atype})")

print("\n--- Equivalent Cypher ---")
print("""
MATCH (a:Agent)
WHERE "summarization" IN a.capabilities
RETURN a.id AS agent, labels(a) AS type
""")

### Query C: Multi-hop — Find documents connected to a specific agent through tasks

In [ ]:
# SPARQL
sparql_c = """
PREFIX ex: <http://example.org/>

SELECT ?doc ?title ?task
WHERE {
    ?task ex:assignedTo ex:AgentSummarizer .
    ?task ex:hasInput ?doc .
    ?doc ex:title ?title .
}
"""

print("SPARQL: Documents connected to AgentSummarizer via tasks")
print("=" * 55)
for row in g.query(sparql_c):
    doc = str(row.doc).split("/")[-1]
    task = str(row.task).split("/")[-1]
    print(f"  {doc}: {row.title}  (via {task})")

print("\n--- Equivalent Cypher ---")
print("""
MATCH (a:Agent {id: "AgentSummarizer"})<-[:ASSIGNED_TO]-(t:Task)-[:HAS_INPUT]->(d:Document)
RETURN d.id AS doc, d.title AS title, t.id AS task
""")

### Query D: Aggregation — Count documents per task

In [ ]:
# SPARQL
sparql_d = """
PREFIX ex: <http://example.org/>

SELECT ?task (COUNT(?doc) AS ?docCount)
WHERE {
    ?task ex:hasInput ?doc .
}
GROUP BY ?task
ORDER BY DESC(?docCount)
"""

print("SPARQL: Count documents per task")
print("=" * 40)
for row in g.query(sparql_d):
    task = str(row.task).split("/")[-1]
    print(f"  {task}: {row.docCount} documents")

print("\n--- Equivalent Cypher ---")
print("""
MATCH (t:Task)-[:HAS_INPUT]->(d:Document)
RETURN t.id AS task, COUNT(d) AS docCount
ORDER BY docCount DESC
""")

### Query E: Find all capabilities across all agents

In [ ]:
# SPARQL
sparql_e = """
PREFIX ex: <http://example.org/>

SELECT ?agent ?capability
WHERE {
    ?agent ex:hasCapability ?capability .
}
ORDER BY ?agent ?capability
"""

print("SPARQL: All agent capabilities")
print("=" * 40)
for row in g.query(sparql_e):
    agent = str(row.agent).split("/")[-1]
    print(f"  {agent}: {row.capability}")

print("\n--- Equivalent Cypher ---")
print("""
MATCH (a:Agent)
UNWIND a.capabilities AS capability
RETURN a.id AS agent, capability
ORDER BY agent, capability
""")

## 3. SPARQL Advanced Features

In [ ]:
# OPTIONAL: Find agents and their tasks (include agents with no tasks)
sparql_optional = """
PREFIX ex: <http://example.org/>

SELECT ?agent ?type ?task ?status
WHERE {
    ?agent a ?type .
    FILTER(?type IN (ex:LLMAgent, ex:RetrievalAgent))
    OPTIONAL {
        ?task ex:assignedTo ?agent .
        ?task ex:hasStatus ?status .
    }
}
"""

print("SPARQL OPTIONAL: Agents with their tasks (LEFT JOIN)")
print("=" * 55)
for row in g.query(sparql_optional):
    agent = str(row.agent).split("/")[-1]
    atype = str(row.type).split("/")[-1]
    task = str(row.task).split("/")[-1] if row.task else "(none)"
    status = str(row.status) if row.status else "-"
    print(f"  {agent} ({atype}) → {task} [{status}]")

In [ ]:
# CONSTRUCT: Create new triples (e.g., infer agent-document relationships)
sparql_construct = """
PREFIX ex: <http://example.org/>

CONSTRUCT {
    ?agent ex:handlesDocument ?doc .
}
WHERE {
    ?task ex:assignedTo ?agent .
    ?task ex:hasInput ?doc .
}
"""

inferred = g.query(sparql_construct)
print("SPARQL CONSTRUCT: Inferred agent-document relationships")
print("=" * 55)
for s, p, o in inferred:
    agent = str(s).split("/")[-1]
    doc = str(o).split("/")[-1]
    print(f"  {agent} —handlesDocument→ {doc}")

## 4. Cypher Advanced Features

These Cypher queries demonstrate features that go beyond basic pattern matching:

### OPTIONAL MATCH (left join)
```cypher
MATCH (a:Agent)
OPTIONAL MATCH (a)<-[:ASSIGNED_TO]-(t:Task)
RETURN a.id AS agent, t.id AS task, t.status AS status
```

### WITH for chaining / piping results
```cypher
MATCH (a:Agent)<-[:ASSIGNED_TO]-(t:Task)
WITH a, COUNT(t) AS taskCount
WHERE taskCount > 1
RETURN a.id AS busyAgent, taskCount
```

### Graph algorithm: Find best agent by past performance
```cypher
MATCH (a:Agent)-[r:HANDLED]->(t:Task)
WHERE t.type = "Summarization"
WITH a, avg(r.qualityScore) AS avgScore, count(*) AS n
WHERE n > 20
RETURN a.id, avgScore
ORDER BY avgScore DESC
LIMIT 3
```

### Shortest path
```cypher
MATCH p = shortestPath(
    (a:Agent {id: "AgentSummarizer"})-[*]-(d:Document {id: "Doc3"})
)
RETURN p
```

## 5. When to Use Which

| Aspect | RDF / SPARQL | LPG / Cypher |
|--------|-------------|-------------|
| **Semantic reasoning** | Built-in RDFS/OWL inference | No native reasoning |
| **Edge properties** | Via reification (verbose) | Native (first-class) |
| **Traversal performance** | Good | Optimized for deep traversals |
| **Standards** | W3C standard | De facto standard |
| **Federation** | SPARQL 1.1 federation | External tooling |
| **Schema** | Explicit via ontologies | Implicit via labels |
| **Best for** | Ontology-driven KGs, compliance, linked data | Operational contexts, analytics, recommendations |

### For Agentic AI Systems

**Use RDF/SPARQL when:**
- You need formal semantics and reasoning (e.g., "if agent has capability X, it can handle tasks requiring X")
- Policy and compliance checks (data access rules as OWL axioms)
- Interoperability across multiple knowledge sources

**Use LPG/Cypher when:**
- Fast multi-hop traversals for context retrieval
- Graph algorithms (PageRank, community detection) for routing
- Real-time operational workloads with rich edge properties (timestamps, scores)

**Hybrid approach (increasingly common):**
- **RDF layer**: Ontologies, semantic metadata, formal reasoning about types and policies
- **LPG layer**: Concrete agent instances, real-time interactions, performance metrics
- Use SPARQL to determine *what type* of agent is needed; use Cypher to pick *which instance* based on load and performance

## Key Takeaways

- Both RDF/SPARQL and LPG/Cypher can model **agent-task-document** relationships
- **SPARQL** excels at semantic queries with reasoning and standards compliance
- **Cypher** excels at traversal performance and graph analytics
- The choice depends on your use case; **hybrid approaches** combine the strengths of both
- In agentic systems: RDF for the **formal world model**, LPG for the **high-speed operational memory**